<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Full%2C_LoRA%2C_and_QLoRA_Fine_Tuning_with_the_H100_SXM_Benchmarking_(Qwen3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

More details in this article: [H100 PCIe vs SXM vs NVL: Best Single-GPU Choice for LLM Fine-Tuning](https://kaitchup.substack.com/p/h100-pcie-vs-sxm-vs-nvl-best-single)

The main purpose of this notebook was to benchmark the H100 SXM for a single-GPU fine-tuning (full SFT, LoRA, and QLoRA). The results (time+memory consumption) are provided in the logs.

PyTorch version: 2.7.1

CUDA 12.8

In [ ]:
import torch, os, multiprocessing
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from trl import SFTTrainer, SFTConfig
set_seed(1234)

compute_dtype = torch.bfloat16
attn_implementation = 'flash_attention_2'

def fine_tune(model_name, batch_size=1, gradient_accumulation_steps=32, LoRA=False, QLoRA=False):

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.eos_token = "<|im_end|>"
    ds_train = load_dataset("allenai/tulu-3-sft-olmo-2-mixture-0225", split="train[:15000]")

    def process(row):
        row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)
        return row

    ds_train = ds_train.map(
        process,
        num_proc= multiprocessing.cpu_count(),
        load_from_cache_file=False,
    )

    print(ds_train[0]['text'])

    ds_train = ds_train.remove_columns(["messages"])

    if QLoRA:
        bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=compute_dtype,
                bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
                  model_name, quantization_config=bnb_config, device_map={"": 0}, attn_implementation=attn_implementation
        )
        model = prepare_model_for_kbit_training(model, gradient_checkpointing_kwargs={'use_reentrant':True})
    else:
        model = AutoModelForCausalLM.from_pretrained(
                  model_name, device_map={"": 0}, torch_dtype=compute_dtype, attn_implementation=attn_implementation
        )
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':True})



    if LoRA or QLoRA:
        peft_config = LoraConfig(
                lora_alpha=32,
                lora_dropout=0.05,
                r=32,
                bias="none",
                task_type="CAUSAL_LM",
                target_modules= ['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj"],
                modules_to_save = ["embed_tokens", "lm_head"]
        )
    else:
      peft_config = None

    if LoRA:
        output_dir = "./LoRA/"
    elif QLoRA:
        output_dir = "./QLoRA/"
    else:
        output_dir = "./FFT/"

    training_arguments = SFTConfig(
        output_dir=output_dir,
        optim="adamw_8bit",
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        log_level="debug",
        save_strategy="no",
        logging_steps=25,
        learning_rate=1e-5,
        bf16 = True,
        max_steps=100,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        dataset_text_field="text",
        max_seq_length=4096,
        padding_free=True,
        report_to="none"
    )

    trainer = SFTTrainer(
          model=model,
          train_dataset=ds_train,
          peft_config=peft_config,
          processing_class=tokenizer,
          args=training_arguments,
    )

    #--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")

    trainer_ = trainer.train()


    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
    used_percentage = round(used_memory         /max_memory*100, 3)
    trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
    print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
    print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
    print(f"Peak reserved memory = {used_memory} GB.")
    print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
    print(f"Peak reserved memory % of max memory = {used_percentage} %.")
    print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
    print("-----")
    #----

In [ ]:
fine_tune("Qwen/Qwen3-8B-Base", batch_size=4, gradient_accumulation_steps=32, LoRA=False, QLoRA=False)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

GPU = NVIDIA H100 80GB HBM3. Max memory = 79.096 GB.
15.26 GB of memory reserved.


skipped Embedding(151936, 4096): 593.5M params
bitsandbytes: will optimize Embedding(151936, 4096) in fp32
skipped: 593.5M params
***** Running training *****
  Num examples = 15,000
  Num Epochs = 1
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 128
  Gradient Accumulation steps = 32
  Total optimization steps = 100
  Number of trainable parameters = 8,190,735,360
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
25,1.512500
50,1.293200
75,1.253400
100,1.243300




Training completed. Do not forget to share your model on huggingface.co/models =)




874.4036 seconds used for training.
14.57 minutes used for training.
Peak reserved memory = 77.646 GB.
Peak reserved memory for training = 62.386 GB.
Peak reserved memory % of max memory = 98.167 %.
Peak reserved memory for training % of max memory = 78.874 %.
-----


In [ ]:
fine_tune("Qwen/Qwen3-8B-Base", batch_size=4, gradient_accumulation_steps=32, LoRA=True, QLoRA=False)

Map (num_proc=160): 100%|##########| 15000/15000 [00:00<?, ? examples/s]

<|im_start|>user
Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.<|im_end|>
<|im_start|>assistant
<think>

</think>

Sure, here's an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:
``` 
# Configure the AWS provider
provider "aws" {
  region = "us-east-1"
}

# Create a security group to allow traffic to the ALB
resource "aws_security_group" "alb_sg" {
  name_prefix = "alb_sg"
  ingress {
    from_port = 80
    to_port = 80
    protocol = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }
}

# Create an ALB and target group
resource "aws_lb" "alb" {
  name               = "example-alb"
  internal           = false
  load_balancer_type = "application"

  subnets = ["subnet-12345678", "subnet-87654321"]

  security_groups = [aws_security_group.alb_sg.id]

  tags = {
    Environment = "production"
  }
}

resource "aws_lb_tar

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 4
The following columns in the Training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: id, source, text. If id, source, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
skipped Embedding(151936, 4096): 593.5M params
bitsandbytes: will optimize Embedding(151936, 4096) in fp32
skipped Embedding(151936, 4096): 1187.0M params
bitsandbytes: will optimize Embedding(151936, 4096) in fp32
skipped: 1187.0M params
***** Running training *****
  Num examples = 15,000
  Num Epochs = 1
  Instantaneous batch

GPU = NVIDIA H100 80GB HBM3. Max memory = 79.096 GB.
17.926 GB of memory reserved.


Step,Training Loss
25,1.706100
50,1.556300
75,1.439100
100,1.389500




Training completed. Do not forget to share your model on huggingface.co/models =)




1104.4884 seconds used for training.
18.41 minutes used for training.
Peak reserved memory = 78.223 GB.
Peak reserved memory for training = 60.297 GB.
Peak reserved memory % of max memory = 98.896 %.
Peak reserved memory for training % of max memory = 76.233 %.
-----


In [ ]:
fine_tune("Qwen/Qwen3-8B-Base", batch_size=4, gradient_accumulation_steps=32, LoRA=False, QLoRA=True)

Map (num_proc=160): 100%|##########| 15000/15000 [00:00<?, ? examples/s]

<|im_start|>user
Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.<|im_end|>
<|im_start|>assistant
<think>

</think>

Sure, here's an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:
``` 
# Configure the AWS provider
provider "aws" {
  region = "us-east-1"
}

# Create a security group to allow traffic to the ALB
resource "aws_security_group" "alb_sg" {
  name_prefix = "alb_sg"
  ingress {
    from_port = 80
    to_port = 80
    protocol = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }
}

# Create an ALB and target group
resource "aws_lb" "alb" {
  name               = "example-alb"
  internal           = false
  load_balancer_type = "application"

  subnets = ["subnet-12345678", "subnet-87654321"]

  security_groups = [aws_security_group.alb_sg.id]

  tags = {
    Environment = "production"
  }
}

resource "aws_lb_tar

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 4
The following columns in the Training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, source, id. If text, source, id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
skipped Embedding(151936, 4096): 593.5M params
bitsandbytes: will optimize Embedding(151936, 4096) in fp32
skipped Embedding(151936, 4096): 1187.0M params
bitsandbytes: will optimize Embedding(151936, 4096) in fp32
skipped: 1187.0M params
***** Running training *****
  Num examples = 15,000
  Num Epochs = 1
  Instantaneous batch

GPU = NVIDIA H100 80GB HBM3. Max memory = 79.096 GB.
14.857 GB of memory reserved.


The input hidden states seems to be silently casted in float32, this might be related to the fact you have upcasted embedding or layer norm layers in float32. We will cast back the input in torch.bfloat16.


Step,Training Loss
25,1.807100
50,1.651400
75,1.552600
100,1.512700




Training completed. Do not forget to share your model on huggingface.co/models =)




1781.1146 seconds used for training.
29.69 minutes used for training.
Peak reserved memory = 75.77 GB.
Peak reserved memory for training = 60.913 GB.
Peak reserved memory % of max memory = 95.795 %.
Peak reserved memory for training % of max memory = 77.011 %.
-----
